# Torghut capital authority

**TL;DR.** This notebook displays `/trading/status` authority fields and reason codes verbatim. The final
`action_authority.entry_allowed` value remains authoritative even when a lower-level execution gate,
broker reconciliation, accounting parity, or capital control reports success. No composite “allowed”
boolean is calculated here.

## Context and method

The source is the GET-only scheduler `/trading/status` endpoint. Each component retains its original payload for audit.

In [ ]:
import json
import math

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import HTML, display
from plotly.subplots import make_subplots

from app.notebook_data import (
    Window,
    adapter_from_environment,
    capital_authority,
    execution_evidence,
    flow_snapshot,
    mode_banner,
    strategy_lifecycle,
)

adapter = globals().get('adapter') or adapter_from_environment()
banner = mode_banner(adapter)
banner_color = '#9a6700' if adapter.mode == 'fixture' else '#b42318'
display(HTML(f'''<div style="padding:14px 18px;border:2px solid {banner_color};border-radius:8px;
  font-weight:700;background:#fff8e6;margin-bottom:14px">{banner}</div>'''))

def bounded_points(frame, series_column, *, time_column='event_ts', preferred=5000):
    # Deterministically thin display points while preserving each series.
    if frame.empty or len(frame) <= preferred:
        return frame.copy()
    series_count = max(1, frame[series_column].nunique())
    per_series = max(2, preferred // series_count)
    chunks = []
    for _, group in frame.sort_values(time_column).groupby(series_column, sort=True):
        stride = max(1, math.ceil(len(group) / per_series))
        chunks.append(group.iloc[::stride])
    result = pd.concat(chunks, ignore_index=True)
    if len(result) > 10000:
        raise ValueError('figure point cap exceeded after deterministic thinning')
    return result

def bounded_rows(frame, *, preferred=5000):
    if frame.empty or len(frame) <= preferred:
        return frame.copy()
    stride = max(1, math.ceil(len(frame) / preferred))
    result = frame.iloc[::stride].head(10000).copy()
    if len(result) > 10000:
        raise ValueError('figure point cap exceeded after deterministic thinning')
    return result

def snapshot_frame(snapshot, name):
    if name not in snapshot.datasets:
        return pd.DataFrame()
    return snapshot.frame(name)

def empty_panel(message):
    display(HTML(f'<div style="padding:18px;border:1px solid #d0d5dd;border-radius:8px;color:#475467">{message}</div>'))

In [ ]:
snapshot = capital_authority(adapter=adapter)
display(snapshot.provenance())
components = snapshot_frame(snapshot, 'components')
raw_status_rows = snapshot.datasets.get('raw_status', ())
raw_status = raw_status_rows[0] if raw_status_rows else {}
action = raw_status.get('action_authority', {})
if not isinstance(action.get('entry_allowed'), bool):
    empty_panel('CAPITAL AUTHORITY UNAVAILABLE — ' + '; '.join(snapshot.messages))
else:
    final_value = action['entry_allowed']
    color = '#027a48' if final_value else '#b42318'
    label = 'ENTRY ALLOWED' if final_value else 'ENTRY BLOCKED'
    display(HTML(f'''<div style="padding:20px;border:4px solid {color};border-radius:10px;
      font-size:24px;font-weight:900">FINAL ACTION AUTHORITY: {label}</div>'''))

## Authority components and verbatim reason codes

In [ ]:
if components.empty:
    empty_panel('Authority components are unavailable.')
else:
    display_table = components.drop(columns=['payload']).copy()
    display_table['reason_codes'] = display_table['reason_codes'].apply(lambda value: json.dumps(value, sort_keys=True))
    display(display_table)

## Raw authoritative action payload

In [ ]:
display(HTML(f'<pre style="white-space:pre-wrap">{json.dumps(action, indent=2, sort_keys=True, default=str)}</pre>')) if action else empty_panel('Authoritative action payload is unavailable.')

## Component payloads

In [ ]:
for row in snapshot.datasets.get('components', ()):
    display(HTML(f"<h3>{row['component']}</h3><pre style='white-space:pre-wrap'>{json.dumps(row['payload'], indent=2, sort_keys=True, default=str)}</pre>"))
if not snapshot.datasets.get('components'):
    empty_panel('Component payloads are unavailable.')

## Takeaways

Use the final action authority and its own reason codes for capital decisions. Lower-level green states are diagnostic inputs, not an override.